# 03 — Knowledge Graph Completion: RESCAL, TransE, SimplE, and HPO

**Goal:** Train KG embedding models on a tiny synthetic knowledge graph
(researchers, papers, institutions, topics) and rank missing links.

**Why this matters:** TGraphX supports tensor-aware KG workflows inside the
same framework as GNN training and graph mining.

**TGraphX subsystem:** `tgraphx.kg`

**Data:** Synthetic — no download required.

**Runtime:** < 60 seconds on CPU.

In [ ]:
import torch
from tgraphx.kg import (
    KnowledgeGraph,
    TransEModel, RESCALModel, SimplEModel,
    evaluate_filtered_ranking,
    list_kg_models,
    run_kg_hpo,
)
print("Available KG models:", list(list_kg_models().keys()))

## 2. Scenario: Academic Knowledge Graph

We model a small academic community:

| Entity type | IDs |
|---|---|
| Researchers | 0–4 |
| Papers | 5–9 |
| Topics | 10–12 |
| Institutions | 13–14 |

Relations:
- 0 = `authored` (researcher → paper)
- 1 = `affiliated_with` (researcher → institution)
- 2 = `covers_topic` (paper → topic)
- 3 = `supervised` (researcher → researcher)

In [ ]:
# Hand-crafted tiny academic KG.
heads = torch.tensor([0, 1, 2, 3, 4, 0, 1, 2, 3, 4,  5, 6, 7, 8, 9,  0, 1])
rels  = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1,  2, 2, 2, 2, 2,  3, 3])
tails = torch.tensor([5, 6, 7, 8, 9, 13,14,13,14,13, 10,11,12,10,11, 1, 2])
N_e, N_r = 15, 4

kg = KnowledgeGraph.from_hrt(heads, rels, tails, num_entities=N_e, num_relations=N_r)
print(f"KG: {kg.num_entities} entities, {kg.num_relations} relations, "
      f"{kg.num_triples} triples")

## 3. Train Individual Models

In [ ]:
def train_eval(model, kg, epochs=40, seed=42):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    triples = kg.triples
    for _ in range(epochs):
        neg = triples.clone()
        neg[:, 2] = torch.randint(0, N_e, (triples.size(0),))
        loss = (1.0 + model.score_triples(neg) - model.score_triples(triples)).clamp(min=0).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    # Evaluate.
    all_pos = set(map(tuple, triples.tolist()))
    res = evaluate_filtered_ranking(model, triples, all_pos, N_e, filtered=True, hits_at=(1, 3))
    return res

for name, cls in [("TransE", TransEModel), ("RESCAL", RESCALModel), ("SimplE", SimplEModel)]:
    model = cls(N_e, N_r, embedding_dim=16)
    res   = train_eval(model, kg)
    print(f"{name:8s}  MRR={res.filt_mrr:.3f}  H@1={res.filt_hits[1]:.3f}  H@3={res.filt_hits[3]:.3f}")

## 4. Why RESCAL Captures Asymmetric Relations

DistMult scores `f(h,r,t) = ⟨h, r, t⟩` which is **symmetric in h and t**:
`f(A, authored, Paper1) == f(Paper1, authored, A)`.

RESCAL uses a matrix per relation: `f(h,r,t) = h^T M_r t`.  A non-symmetric
`M_r` means the model can correctly score directed relations.

SimplE also captures asymmetry via separate forward/inverse embeddings.

In [ ]:
# Demonstrate asymmetry: RESCAL can distinguish direction.
rescal = RESCALModel(N_e, N_r, embedding_dim=8)
fwd = rescal.score_triples(torch.tensor([[0, 0, 5]]))  # researcher authored paper
rev = rescal.score_triples(torch.tensor([[5, 0, 0]]))  # paper authored researcher?
print(f"Forward (researcher→paper): {fwd.item():.4f}")
print(f"Reverse (paper→researcher): {rev.item():.4f}")
print("Scores differ (random init): asymmetry can be learned.")

## 5. KG HPO — Grid Search Over Models and Hyper-Params

In [ ]:
result = run_kg_hpo(
    kg,
    model_names=["TransE", "DistMult", "SimplE"],
    search_space={
        "embedding_dim": [8, 16],
        "lr": [1e-2, 5e-3],
    },
    metric="mrr",
    strategy="grid",
    max_trials=6,
    epochs=20,
    seed=42,
)
print("Best model:", result.best_model_name)
print("Best config:", result.best_config)
print("Best MRR:  ", result.best_metrics["mrr"])
print(f"\nTrials run: {len(result.trials)}")

In [ ]:
result.summary()

## 6. Next Steps

- **Tutorial:** `tutorials/kg_benchmark_quickstart.py`
- **API ref:** `tgraphx/kg/` — TrainE, DistMult, ComplEx, RotatE, RESCAL, SimplE
- **Limitations:** this is a tiny 17-triple KG.  Filtered MRR is not a
  reliable metric at this scale.  For meaningful KG benchmarks use
  FB15k-237 or WN18RR (optional PyG adapter; explicit download required).